# Notebook 6.1  Fine-tuning a speech foundation model for Arabic: full vs LoRA

**Companion to Chapter 6, *Introduction to Arabic Speech Technologies*.**

**Goal.** Build intuition for self-supervised objectives and for the cost of adaptation, with no
downloads. We illustrate the contrastive and masked-prediction ideas on toy vectors, then compute
the trainable-parameter math for full fine-tuning versus LoRA. A real XLS-R fine-tuning cell is
included but commented out for Colab. An Exercise solutions section follows.

## 1. Self-supervised objectives on toy vectors

Contrastive (wav2vec 2.0): from a masked position's context, pick the true latent against distractors.
Masked prediction (HuBERT): predict a discrete cluster id for the masked frame. Both turn unlabeled
data into a training signal.

In [ ]:
import numpy as np
rng=np.random.default_rng(0)
# 8 frames, 4-dim latents; mask frame 3 and score candidates by similarity to its context
lat=rng.normal(size=(8,4))
context=(lat[2]+lat[4])/2                 # neighbours of masked frame 3
true=lat[3]; distractors=lat[[0,1,5,6]]
cands=np.vstack([true,distractors])
sims=cands@context
print('contrastive: picked candidate index', int(np.argmax(sims)), '(0 = the true latent)')
# masked prediction: cluster latents, then the target for a masked frame is its cluster id
from numpy import argmin
centroids=lat[[0,3]]                       # two toy clusters
clusters=[int(argmin([np.sum((f-c)**2) for c in centroids])) for f in lat]
print('masked-prediction target for frame 3 is cluster', clusters[3])

## 2. Adaptation cost: full fine-tuning vs LoRA

In [ ]:
def param_counts(d_model=1024, n_layers=24, lora_rank=8, adapt_projections=2):
    total = 12 * n_layers * d_model**2     # rough Transformer parameter estimate
    lora = adapt_projections * n_layers * 2 * d_model * lora_rank
    return total, lora
total, lora = param_counts()
print(f'encoder parameters (approx):      {total/1e6:6.1f} M')
print(f'full fine-tuning trains:          {total/1e6:6.1f} M (100%)')
print(f'LoRA (rank 8, Q and V) trains:    {lora/1e6:6.3f} M ({100*lora/total:.3f}%)')
print(f'=> LoRA trains about {total/lora:.0f}x fewer parameters.')

## 3. Optional: real fine-tuning on Colab (commented)

Uncomment in Colab with a GPU to fine-tune XLS-R on a small Arabic set and compare full vs LoRA.

In [ ]:
# !pip -q install transformers peft datasets torchaudio jiwer
# from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
# from peft import LoraConfig, get_peft_model
# model = Wav2Vec2ForCTC.from_pretrained('facebook/wav2vec2-xls-r-300m')
# lora = LoraConfig(r=8, target_modules=['q_proj','v_proj'])
# model = get_peft_model(model, lora)        # train only the LoRA parameters
# model.print_trainable_parameters()
print('See the commented cell above to run real XLS-R fine-tuning on Colab.')

## 4. Exercise solutions

**Exercise 1.** wav2vec 2.0 masks latent spans and contrastively picks the true quantized latent against distractors; HuBERT masks spans and predicts a discrete cluster id; contrastive predictive coding predicts future latents against negatives. All three create supervision from the audio itself (see the toy demo in Section 1).

In [ ]:
# Exercise 2: parameter ratio for a ~300M encoder, recomputed
total, lora = param_counts(d_model=1024, n_layers=24, lora_rank=8)
print(f'full: {total/1e6:.0f} M   LoRA: {lora/1e6:.3f} M   ratio ~ {total/lora:.0f}x fewer trained')

**Exercise 3 (design).** Freeze the encoder; train a small probe on one layer to predict the phone, and a separate probe to predict the speaker; report per-layer accuracy. If phone accuracy peaks in the middle layers while speaker accuracy is higher in early layers, the two are encoded at different depths.

**Exercise 4 (design).** Start from a multilingual XLS-R encoder; continually pretrain on the 700 unlabeled hours to move it toward Arabic; then adapt on the 10 labeled hours with LoRA or adapters (safer than full fine-tuning with so little data), reporting per-dialect error on speaker-disjoint splits.

**Exercise 5 (design).** Report per-dialect WER and CER on dialectal sets (SADA, MASC) alongside read benchmarks (Common Voice, FLEURS) and MSA broadcast (MGB-2), under one published normalization; detect MSA bias by counting outputs that replace a dialectal word with its formal equivalent.